# Train Dual-Branch Gated Fusion

Loads `cores_features.npz` + `xlsr_features.npz`, trains experts + input-conditioned gate with:
- Cross-entropy (label smoothing 0.15)
- Energy margin loss
- Gate diversity KL + entropy
- Gate frozen for first 10 epochs

**Stub SSL mode:** if XLSR cache is missing, builds fake 1024-d vectors (home smoke test only).

See [`docs/RUNBOOK.md`](../docs/RUNBOOK.md).

## Cell 1 — Setup

In [ ]:
!pip install -q tqdm

import json
import random
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/mlaad-dual-branch')
else:
    ROOT = Path('/content/mlaad-dual-branch')

ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / 'cache').mkdir(exist_ok=True)
(ROOT / 'checkpoints').mkdir(exist_ok=True)
print('Working dir:', ROOT)

## Cell 2 — Config (paper Section 2.4–2.5)

In [ ]:
@dataclass
class Config:
  cores_dim: int = 66
  ssl_dim: int = 1024
  expert_hidden: int = 512
  expert_out: int = 256
  gate_hidden: int = 128
  dropout_expert: float = 0.3
  dropout_gate: float = 0.2
  num_classes: int = 24

  batch_size: int = 64  # use 128 on uni GPU for paper settings
  lr: float = 1e-4
  weight_decay: float = 1e-4
  epochs: int = 30  # use 150 on uni for paper settings
  gate_freeze_epochs: int = 10
  label_smoothing: float = 0.15
  grad_clip: float = 5.0
  min_lr: float = 5e-6

  lambda_e: float = 0.5
  lambda_g: float = 0.05
  lambda_h: float = 0.3
  m_in: float = -15.0
  m_out: float = -2.0

  seed: int = 42
  allow_stub_ssl: bool = True  # set False at uni once xlsr cache exists

  cores_cache: str = str(ROOT / 'cache' / 'cores_features.npz')
  xlsr_cache: str = str(ROOT / 'cache' / 'xlsr_features.npz')
  ckpt_path: str = str(ROOT / 'checkpoints' / 'dual_branch_best.pt')


cfg = Config()

def set_seed(seed: int):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print(asdict(cfg))

## Cell 3 — Load / join caches (stub SSL if needed)

In [ ]:
def load_npz(path: Path) -> Dict:
  loaded = np.load(path, allow_pickle=True)
  return {k: loaded[k] for k in loaded.files}


def make_stub_ssl_from_cores(cores: Dict, ssl_dim: int = 1024) -> Dict:
  """Deterministic fake SSL features aligned to CORES utt_ids (home only)."""
  print('WARNING: building STUB x_ssl — not for paper metrics')
  n = len(cores['utt_ids'])
  rng = np.random.RandomState(0)
  # class-conditioned noise so HC-only signal still exists somehow
  x_ssl = rng.randn(n, ssl_dim).astype(np.float32) * 0.1
  for i, lab in enumerate(cores['label_ids']):
    if lab >= 0:
      x_ssl[i, lab % ssl_dim] += 1.0
  return {
      'utt_ids': cores['utt_ids'],
      'splits': cores['splits'],
      'label_ids': cores['label_ids'],
      'is_ood': cores['is_ood'],
      'x_ssl': x_ssl,
  }


def join_caches(cores: Dict, ssl: Dict) -> Dict:
  ssl_map = {uid: i for i, uid in enumerate(ssl['utt_ids'].tolist())}
  idxs_c, idxs_s = [], []
  for i, uid in enumerate(cores['utt_ids'].tolist()):
    if uid in ssl_map:
      idxs_c.append(i)
      idxs_s.append(ssl_map[uid])
  if not idxs_c:
    raise RuntimeError('No overlapping utt_ids between CORES and XLSR caches')
  idxs_c = np.asarray(idxs_c)
  idxs_s = np.asarray(idxs_s)

  # sanity on labels
  if not np.array_equal(cores['label_ids'][idxs_c], ssl['label_ids'][idxs_s]):
    raise RuntimeError('label_ids mismatch on overlapping utt_ids')

  joined = {
      'utt_ids': cores['utt_ids'][idxs_c],
      'splits': cores['splits'][idxs_c],
      'label_ids': cores['label_ids'][idxs_c],
      'is_ood': cores['is_ood'][idxs_c],
      'x_hc': cores['x_hc'][idxs_c].astype(np.float32),
      'x_ssl': ssl['x_ssl'][idxs_s].astype(np.float32),
  }
  print(f'Joined utterances: {len(joined["utt_ids"])} '
        f'(cores={len(cores["utt_ids"])}, ssl={len(ssl["utt_ids"])})')
  return joined


cores_path = Path(cfg.cores_cache)
xlsr_path = Path(cfg.xlsr_cache)

if not cores_path.exists():
  raise FileNotFoundError(
      f'Missing {cores_path}. Run CORES notebook first (or copy cache here).')

cores = load_npz(cores_path)
if xlsr_path.exists():
  ssl = load_npz(xlsr_path)
  print('Loaded real XLSR cache')
elif cfg.allow_stub_ssl:
  ssl = make_stub_ssl_from_cores(cores, cfg.ssl_dim)
else:
  raise FileNotFoundError(f'Missing {xlsr_path} and stub SSL disabled')

data = join_caches(cores, ssl)
print('x_hc', data['x_hc'].shape, '| x_ssl', data['x_ssl'].shape)

## Cell 4 — Normalize (train ID stats only)

In [ ]:
def norm_stats(x: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
  mean = x[mask].mean(axis=0).astype(np.float32)
  std = x[mask].std(axis=0).astype(np.float32)
  std[std < 1e-6] = 1.0
  return mean, std

train_mask = (data['splits'] == 'train') & (~data['is_ood']) & (data['label_ids'] >= 0)
hc_mean, hc_std = norm_stats(data['x_hc'], train_mask)
ssl_mean, ssl_std = norm_stats(data['x_ssl'], train_mask)

x_hc = (data['x_hc'] - hc_mean) / hc_std
x_ssl = (data['x_ssl'] - ssl_mean) / ssl_std
print('Normalized. train ID count:', int(train_mask.sum()))

## Cell 5 — Dataset / loaders

In [ ]:
class DualDataset(Dataset):
  def __init__(self, x_hc, x_ssl, y, is_ood):
    self.x_hc = torch.from_numpy(x_hc).float()
    self.x_ssl = torch.from_numpy(x_ssl).float()
    self.y = torch.from_numpy(y).long()
    self.is_ood = torch.from_numpy(is_ood.astype(np.bool_))

  def __len__(self):
    return len(self.y)

  def __getitem__(self, idx):
    return self.x_hc[idx], self.x_ssl[idx], self.y[idx], self.is_ood[idx]


def split_mask(split_name: str, id_only: bool = False):
  m = data['splits'] == split_name
  if id_only:
    m &= (~data['is_ood']) & (data['label_ids'] >= 0)
  return m


# Train: ID samples for CE + Dev-OOD rows that landed in train? Paper uses train ID + aux Dev-OOD.
# We include train ID always, and any is_ood=True from 'dev' as OOD aux during training.
train_id_m = split_mask('train', id_only=True)
dev_ood_m = (data['splits'] == 'dev') & (data['is_ood'])
train_m = train_id_m | dev_ood_m

dev_id_m = split_mask('dev', id_only=True)
dev_all_m = data['splits'] == 'dev'  # ID + OOD for FPR95 during training

train_loader = DataLoader(
    DualDataset(x_hc[train_m], x_ssl[train_m], data['label_ids'][train_m], data['is_ood'][train_m]),
    batch_size=cfg.batch_size, shuffle=True, drop_last=False)
dev_loader = DataLoader(
    DualDataset(x_hc[dev_all_m], x_ssl[dev_all_m], data['label_ids'][dev_all_m], data['is_ood'][dev_all_m]),
    batch_size=cfg.batch_size, shuffle=False)

print('Train rows:', int(train_m.sum()), '(ID', int(train_id_m.sum()), '+ Dev-OOD', int(dev_ood_m.sum()), ')')
print('Dev rows:', int(dev_all_m.sum()), '(ID', int(dev_id_m.sum()), ')')

## Cell 6 — Models (experts + gate + classifier)

In [ ]:
class ExpertMLP(nn.Module):
  def __init__(self, in_dim, hidden=512, out=256, dropout=0.3):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.BatchNorm1d(hidden),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(hidden, out),
        nn.BatchNorm1d(out),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class GatingNetwork(nn.Module):
  def __init__(self, in_dim=512, hidden=128, dropout=0.2):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(hidden, 2),
    )

  def forward(self, e_hc, e_ssl):
    logits = self.net(torch.cat([e_hc, e_ssl], dim=-1))
    alpha = torch.softmax(logits, dim=-1)  # (B, 2) -> [alpha_hc, alpha_ssl]
    return alpha


class DualBranchModel(nn.Module):
  def __init__(self, cfg: Config):
    super().__init__()
    self.expert_hc = ExpertMLP(cfg.cores_dim, cfg.expert_hidden, cfg.expert_out, cfg.dropout_expert)
    self.expert_ssl = ExpertMLP(cfg.ssl_dim, cfg.expert_hidden, cfg.expert_out, cfg.dropout_expert)
    self.gate = GatingNetwork(cfg.expert_out * 2, cfg.gate_hidden, cfg.dropout_gate)
    self.classifier = nn.Linear(cfg.expert_out, cfg.num_classes)

  def forward(self, x_hc, x_ssl):
    e_hc = self.expert_hc(x_hc)
    e_ssl = self.expert_ssl(x_ssl)
    alpha = self.gate(e_hc, e_ssl)
    e_fused = alpha[:, 0:1] * e_hc + alpha[:, 1:2] * e_ssl
    logits = self.classifier(e_fused)
    return logits, alpha, e_hc, e_ssl, e_fused


model = DualBranchModel(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Dual-branch params: {n_params:,}')

with torch.no_grad():
  logits, alpha, *_ = model(
      torch.randn(4, cfg.cores_dim, device=DEVICE),
      torch.randn(4, cfg.ssl_dim, device=DEVICE))
  print('logits', logits.shape, '| alpha', alpha.shape, '| alpha sum', alpha.sum(dim=-1))

## Cell 7 — Losses

In [ ]:
def energy_from_logits(logits: torch.Tensor) -> torch.Tensor:
  # E(x) = -logsumexp(z)
  return -torch.logsumexp(logits, dim=-1)


def label_smoothed_ce(logits, targets, num_classes, smoothing=0.15):
  log_probs = F.log_softmax(logits, dim=-1)
  with torch.no_grad():
    true_dist = torch.zeros_like(log_probs)
    true_dist.fill_(smoothing / (num_classes - 1))
    true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
  return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))


def energy_margin_loss(logits, is_ood, m_in, m_out):
  E = energy_from_logits(logits)
  id_mask = ~is_ood
  ood_mask = is_ood
  loss = logits.new_tensor(0.0)
  if id_mask.any():
    loss = loss + F.relu(E[id_mask] - m_in).mean()
  if ood_mask.any():
    loss = loss + F.relu(m_out - E[ood_mask]).mean()
  return loss


def gate_diversity_loss(alpha, is_ood):
  """L_gate = -KL(mean_alpha_id || mean_alpha_ood)"""
  id_mask = ~is_ood
  ood_mask = is_ood
  if not (id_mask.any() and ood_mask.any()):
    return alpha.new_tensor(0.0)
  a_id = alpha[id_mask].mean(dim=0).clamp_min(1e-8)
  a_ood = alpha[ood_mask].mean(dim=0).clamp_min(1e-8)
  # KL(p||q) = sum p log(p/q)
  kl = torch.sum(a_id * (torch.log(a_id) - torch.log(a_ood)))
  return -kl


def gate_entropy_loss(alpha):
  # Lent = sum alpha log alpha  (minimize this? paper adds λh * Lent;
  # maximizing entropy means minimizing sum alpha log alpha which is negative)
  # Paper: Lent = Σ αk log αk and includes + λh Lent in total loss,
  # with λh=0.3 to prevent one-hot collapse (encourage higher entropy → more negative Lent).
  a = alpha.clamp_min(1e-8)
  return torch.sum(a * torch.log(a), dim=-1).mean()


def total_loss(logits, alpha, y, is_ood, cfg: Config):
  id_mask = (~is_ood) & (y >= 0)
  losses = {}
  if id_mask.any():
    losses['ce'] = label_smoothed_ce(logits[id_mask], y[id_mask], cfg.num_classes, cfg.label_smoothing)
  else:
    losses['ce'] = logits.new_tensor(0.0)
  losses['energy'] = energy_margin_loss(logits, is_ood, cfg.m_in, cfg.m_out)
  losses['gate'] = gate_diversity_loss(alpha, is_ood)
  losses['ent'] = gate_entropy_loss(alpha)
  losses['total'] = (
      losses['ce']
      + cfg.lambda_e * losses['energy']
      + cfg.lambda_g * losses['gate']
      + cfg.lambda_h * losses['ent']
  )
  return losses

## Cell 8 — Metrics helpers (Dev FPR95 for checkpointing)

In [ ]:
def sme_score(logits: torch.Tensor) -> torch.Tensor:
  """Softmax Energy: energy after softmax. Lower (more negative) => more ID-like."""
  probs = torch.softmax(logits, dim=-1)
  return -torch.logsumexp(torch.log(probs.clamp_min(1e-12)), dim=-1)


@torch.no_grad()
def eval_dev(model, loader, cfg: Config):
  model.eval()
  all_logits, all_y, all_ood, all_alpha = [], [], [], []
  for xh, xs, y, ood in loader:
    xh, xs = xh.to(DEVICE), xs.to(DEVICE)
    logits, alpha, *_ = model(xh, xs)
    all_logits.append(logits.cpu())
    all_y.append(y)
    all_ood.append(ood)
    all_alpha.append(alpha.cpu())
  logits = torch.cat(all_logits)
  y = torch.cat(all_y)
  ood = torch.cat(all_ood).bool()
  alpha = torch.cat(all_alpha)

  id_mask = (~ood) & (y >= 0)
  id_acc = 0.0
  if id_mask.any():
    id_acc = (logits[id_mask].argmax(-1) == y[id_mask]).float().mean().item()

  scores = sme_score(logits).numpy()
  id_scores = scores[id_mask.numpy()]
  ood_scores = scores[ood.numpy()]

  fpr95 = 1.0
  if len(id_scores) and len(ood_scores):
    # threshold at 95% ID TPR: ID should have lower scores
    thr = np.percentile(id_scores, 95)
    # ID accepted if score <= thr; FPR95 = fraction of OOD accepted
    fpr95 = float((ood_scores <= thr).mean())

  # EERc: ID sample correct only if correctly classified AND accepted as ID
  eerc = 1.0
  if len(id_scores) and len(ood_scores):
    thr = np.percentile(id_scores, 95)
    preds = logits.argmax(-1)
    correct = (preds == y) & id_mask
    accepted = torch.from_numpy(scores <= thr) & id_mask
    # joint error among ID: wrong class OR rejected
    id_error = (~(correct & accepted))[id_mask].float().mean().item()
    # also count OOD accepted as error contribution (simplified joint)
    ood_accept = float((ood_scores <= thr).mean())
    eerc = 0.5 * (id_error + ood_accept)

  alpha_id = alpha[id_mask].mean(0).tolist() if id_mask.any() else [0.5, 0.5]
  alpha_ood = alpha[ood].mean(0).tolist() if ood.any() else [0.5, 0.5]
  return {
      'id_acc': id_acc,
      'fpr95': fpr95,
      'eerc': eerc,
      'alpha_id': alpha_id,
      'alpha_ood': alpha_ood,
  }

## Cell 9 — Train (gate freeze → full)

In [ ]:
def set_gate_trainable(model, trainable: bool):
  for p in model.gate.parameters():
    p.requires_grad = trainable


optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=cfg.min_lr)

best_fpr95 = float('inf')
history = []

for epoch in range(1, cfg.epochs + 1):
  # Gate freeze schedule
  gate_on = epoch > cfg.gate_freeze_epochs
  set_gate_trainable(model, gate_on)
  # rebuild optimizer param groups when gate unfreezes (once)
  if epoch == cfg.gate_freeze_epochs + 1:
    optimizer = torch.optim.AdamW(model.parameters(), lr=optimizer.param_groups[0]['lr'],
                                  weight_decay=cfg.weight_decay)
    print(f'Epoch {epoch}: gate UNFROZEN')

  model.train()
  # keep gate in eval-ish? No — when frozen, still forward; grads blocked via requires_grad
  running = {'total': 0.0, 'ce': 0.0, 'energy': 0.0, 'gate': 0.0, 'ent': 0.0}
  n_batches = 0
  for xh, xs, y, ood in train_loader:
    xh, xs, y, ood = xh.to(DEVICE), xs.to(DEVICE), y.to(DEVICE), ood.to(DEVICE)
    optimizer.zero_grad()
    logits, alpha, *_ = model(xh, xs)
    losses = total_loss(logits, alpha, y, ood, cfg)
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    optimizer.step()
    for k in running:
      running[k] += float(losses[k].detach().cpu())
    n_batches += 1
  scheduler.step()

  for k in running:
    running[k] /= max(n_batches, 1)

  metrics = eval_dev(model, dev_loader, cfg)
  row = {'epoch': epoch, 'gate_on': gate_on, **running, **metrics}
  history.append(row)

  if metrics['fpr95'] < best_fpr95:
    best_fpr95 = metrics['fpr95']
    torch.save({
        'model': model.state_dict(),
        'cfg': asdict(cfg),
        'hc_mean': hc_mean, 'hc_std': hc_std,
        'ssl_mean': ssl_mean, 'ssl_std': ssl_std,
        'epoch': epoch,
        'metrics': metrics,
    }, cfg.ckpt_path)

  if epoch == 1 or epoch % 5 == 0 or epoch == cfg.epochs:
    print(
        f"Epoch {epoch:03d} gate={'ON' if gate_on else 'OFF'} | "
        f"loss {running['total']:.3f} ce {running['ce']:.3f} | "
        f"dev ID {metrics['id_acc']:.3f} FPR95 {metrics['fpr95']:.3f} EERc {metrics['eerc']:.3f} | "
        f"a_ssl ID/OOD {metrics['alpha_id'][1]:.3f}/{metrics['alpha_ood'][1]:.3f}"
    )

print('Best Dev FPR95:', best_fpr95)
print('Checkpoint:', cfg.ckpt_path)

## Cell 10 — Save history

In [ ]:
hist_path = ROOT / 'checkpoints' / 'train_history.json'
# convert numpy-ish to plain python
safe = []
for r in history:
  safe.append({k: (v if not isinstance(v, (np.floating, np.integer)) else float(v)) for k, v in r.items()})
with open(hist_path, 'w', encoding='utf-8') as f:
  json.dump(safe, f, indent=2)
print('Wrote', hist_path)